LWA observations are controlled through session definition files (SDFs).  Each file contains a single session that defines the beamformer output and any transient buffer (TBT/TBS) observations.  Although each session uses a single beam, there can be multiple observations per session.

The SDF has four basic parts:

1) observer information, 

2) project information, 

3) session Information, and

4) Observational setup.

These four parts are implemented in LSL in the lsl.common.sdf module.  

To create an object to hold information about an observer:

In [2]:
from lsl.common import sdf

obs = sdf.Observer("Jayce Dowell", 99)

print(obs)

Once an observer is defined, you can create the objects that will hold the project information.  The only required information is the Observer object, project name, and project code.

In [4]:
proj = sdf.Project(obs, "This is a LWA1 project", "COMJD")

print(proj)

Next, the session needs to be created and added to the Project:

In [5]:
ses = sdf.Session("This is a session", 101)
proj.append(ses)

print(ses)

The Session object has a variety of parameters that can be set on it that control the session-wide setup.  This includes which beam the observation runs on and whether or not the DR spectrometer is used. To set the beam and spectrometer mode:

In [7]:
# Set the DRX beam to 3
ses.drx_beam = 3

# Set the spectrometer setup to 1,024 channels, 768 windows per integration, and the Stokes IV mode
ses.spectrometer_channels = 1024
ses.spectrometer_integration = 768
ses.spectrometer_metatag = 'Stokes=IV'

At this point the only thing missing are the actual observations.  To define a beamforming observation that tracks a point on the sky, use the DRX object:

In [ ]:
radec = sdf.DRX("Observation1", "M87", "2026/1/1 18:00:00", "00:10:00.000", 
                12.5137, 12.3911, 37.9e6, 74.03e6, 7)
print(radec)

ses.append(radec)

TRK_RADEC Obs. of 'Observation1':
 Start UTC 2026/01/01 18:00:00.000000
 Duration 0:10:00.000
 Filter: 7
 Frequency: 37899999.990; 74029999.992 Hz
 RA: 12.513700 hr
 Dec. 12.391100 d



The transient buffer streaming (TBS) and time-domain (TBT) modes record all-dipole data and are a distinct observing mode from the beamformer: they live in their own SDF, not alongside DRX observations.  A TBS observation captures a narrow band of channelized voltages at a chosen center frequency; filter codes 7, 8, and 9 select 100, 200, and 300 kHz of bandwidth, respectively:

In [ ]:
# A separate project/session dedicated to a TBS observation.
tbs_proj = sdf.Project(obs, "TBS companion capture", "COMJDT")
tbs_ses = sdf.Session("TBS session", 102)
tbs_proj.append(tbs_ses)

# 30 s of TBS data centered at 38 MHz with 200 kHz of bandwidth (filter code 8).
tbs = sdf.TBS("TBS dump", "M87 field", "2026/1/1 18:00:00", "00:00:30.000",
              38.0e6, 8)
tbs_ses.append(tbs)

print("Is Valid?", tbs_proj.validate())

The session and observation information stored within a Project can also be directly accessed:

In [9]:
for session in proj.sessions:
    print("Session ID:", session.id)
    for i,obs in enumerate(session.observations):
        print("Observation ID:", i)
        print("Is Valid?", obs.validate())
        print(obs)

Session ID: 101
Observation ID: 0
Is Valid? True
TRK_RADEC Obs. of 'Observation1':
 Start UTC 2026/01/01 18:00:00.000000
 Duration 0:10:00.000
 Filter: 7
 Frequency: 37899999.990; 74029999.992 Hz
 RA: 12.513700 hr
 Dec. 12.391100 d



Now all that is left is to create the SDF:

In [ ]:
print(proj.render())

Interferometer Definition Files (IDFs) that are used for scheduling observations with the LWA single baseline interferometer work the same way.  Just import the `idf` from from `lsl.common`.

In [11]:
from lsl.common import idf

The biggest difference between making IDFs and making SDFs is the terminology.  Instead of Sessions there are Runs and instead of Observations there are scans.

# Additional Info
In addition to the modules provided in LSL there are also a variety of scripts in the [session schedules repo](https://github.com/lwa-project/session_schedules) to help build schedules.
